In [1]:
import cv2 as cv

In [11]:
import json
import os
from pathlib import Path
import numpy as np

# Load ground truth annotations
gt_path = "data/belgiumts/sequences_jpg/seq01_01_label_studio_out_coco.json"
with open(gt_path, 'r') as f:
    gt_coco = json.load(f)

# Load detection/tracking annotations
dt_path = "data/out_belgiumts_v3/dt-kursova-seq01_01-False-track1-rawbox-minlen2.json"
with open(dt_path, 'r') as f:
    dt_coco = json.load(f)

# Image directory
img_dir = "data/belgiumts/sequences_jpg"

# Create image id to filename mapping
img_id_to_filename = {img['id']: img['file_name'] for img in gt_coco['images']}

# Group annotations by image_id
gt_by_image = {}
for ann in gt_coco['annotations']:
    img_id = ann['image_id']
    if img_id not in gt_by_image:
        gt_by_image[img_id] = []
    gt_by_image[img_id].append(ann)

dt_by_image = {}
for ann in dt_coco:
    img_id = ann['image_id']
    if img_id not in dt_by_image:
        dt_by_image[img_id] = []
    dt_by_image[img_id].append(ann)

print(f"Loaded {len(gt_coco['images'])} images")
print(f"GT annotations: {len(gt_coco['annotations'])}")
print(f"DT annotations: {len(dt_coco)}")


Loaded 3001 images
GT annotations: 1582
DT annotations: 2150


In [12]:
# Get the first image to determine video properties
first_img_id = sorted(img_id_to_filename.keys())[0]
first_img_path = os.path.join(img_dir, img_id_to_filename[first_img_id])
first_frame = cv.imread(first_img_path)
height, width = first_frame.shape[:2]

print(f"Video resolution: {width}x{height}")

# Setup video writer
output_path = "data/out_visualize_seq01_best.mp4"
fourcc = cv.VideoWriter_fourcc(*'mp4v')
fps = 10  # Adjust as needed
video_writer = cv.VideoWriter(output_path, fourcc, fps, (width, height))

# Process each image in order
for img_id in sorted(img_id_to_filename.keys()):
    filename = img_id_to_filename[img_id]
    img_path = os.path.join(img_dir, filename)
    
    # Read image
    frame = cv.imread(img_path)
    if frame is None:
        print(f"Warning: Could not read {img_path}")
        continue
    
    # Draw ground truth boxes in GREEN
    if img_id in gt_by_image:
        for ann in gt_by_image[img_id]:
            bbox = ann['bbox']  # [x, y, width, height]
            x, y, w, h = [int(v) for v in bbox]
            cv.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)  # GREEN for GT
            # Optionally add label
            cv.putText(frame, 'GT', (x, y - 5), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    
    # Draw detection/tracking boxes in RED
    if img_id in dt_by_image:
        for ann in dt_by_image[img_id]:
            bbox = ann['bbox']  # [x, y, width, height]
            x, y, w, h = [int(v) for v in bbox]
            cv.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)  # RED for detections
            # Optionally add track_id if available
            if 'track_id' in ann:
                cv.putText(frame, f"ID:{ann['track_id']}", (x, y + h + 15), 
                          cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    
    # Add legend
    cv.putText(frame, 'Green: Ground Truth', (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(frame, 'Red: Detections/Tracks', (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    cv.putText(frame, f'Frame: {filename}', (10, height - 20), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    # Write frame to video
    video_writer.write(frame)

# Release video writer
video_writer.release()

print(f"Video saved to: {output_path}")


Video resolution: 1628x1236
Video saved to: data/out_visualize_seq01_best.mp4
